In [ ]:
# NOTEBOOK NAME
# BARRAwindAdder.ipynb
# NOTEBOOK NAME

# # OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr
# from datetime import dat?etime

# # for interpolation
# from scipy.interpolate import interp1d

# # used to play with pathnames to save 
# from pathlib import Path      

# # SPECIAL METHOD TO IMPORT LEROI RADAR GRIDDING PACKAGE AND CUSTOM FUNCTIONS FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
from CustomFunctions1 import *
# from custom_elevation import fetch_srtm, fetch_gebco_local

# # for projecting radar coordinates to lat and lon
# from pyproj import Geod

# # for making cool topo maps
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from matplotlib.colors import ListedColormap, BoundaryNorm

# # for adding lat/lon gridlines on plots
# import matplotlib.ticker as mticker
# from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# for adding a colourful topo base map to the CAPI plots
from custom_elevation import fetch_srtm, fetch_gebco_local
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

In [ ]:
# TRY LOADING IN AN EXAMPLE RADAR FILE


RadarGridsFolder = 'CompressedRadarGrids'
RadarIDno = '22'
RadarYear = 2024
RadarMonth = 2
RadarDay = 14

YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

houri = 12
mini = 0

RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
# add a string of format hh:mm:ss for printing
RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 

print('working on ' + RadarFileTimePrint)

NetCDFstoragePath = ('/scratch/v46/sg3241/tmp/NetCDFs/' + RadarGridsFolder + '/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
                                                         + RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')
# try to load in the netcdf file and if it doesn't work, just keep going through the loop
# try:
xgrid = xr.open_dataset(NetCDFstoragePath)
# except FileNotFoundError:
#     print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
#     continue

In [ ]:
xgrid.time

In [ ]:
# collect the limits for where to grab BARRA data from with a 0.2 degree buffer around the radar data
# and buffer the time by an hour on either side

BarraLatMax = np.max(np.array(xgrid.lat)) + 0.2
BarraLatMin = np.min(np.array(xgrid.lat)) - 0.2

BarraLonMax = np.max(np.array(xgrid.lon)) + 0.2
BarraLonMin = np.min(np.array(xgrid.lon)) - 0.2

BarraTimeMax = np.array(xgrid.time)[0] + np.timedelta64(1, 'h')
BarraTimeMin = np.array(xgrid.time)[0] - np.timedelta64(1, 'h')

In [ ]:
# THIS BLOCK LOADS IN A MONTH-LONG HOURLY BARRA DATA BLOCK FOR TEMPERATURE AT ALL* (MOST) PRESSURE LEVELS

# choose the year and the month you want to look at data for
year  = RadarYear
month = RadarMonth

# # add leading zeros if neccessary to the year and month strings
# YYYY = str(year).zfill(4)
# MM   = str(month).zfill(2)

# pressure levels with temperature variables associated with them
PLevels = [1000, 925, 850, 700, 600, 500, 400, 300, 200] #, #950] # the kernel sometimes crashes when I include 200 hPa
# there is no geopotential height data for 950 hPa, but there is temperature data

# create an empty data structure carrying-bag
TBarraDataStructs = {}

# loop through each pressure level and grab a lat-lon slice over Queensland 
for PLevel in PLevels:
    
    Tvariable = 'ta' + str(PLevel)

    # where the BARRA data live
    TBARRAfolder = f'/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/{Tvariable}/latest/'
    TBARRAfile = Tvariable + f'_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_1hr_{YYYY}{MM}-{YYYY}{MM}.nc'
    TBARRApath = TBARRAfolder + TBARRAfile

    Tdata = xr.open_dataset(TBARRApath)

    # take that Queensland slice
    Tdata = Tdata.sel(
        lat=slice(BarraLatMin, BarraLatMax),
        lon=slice(BarraLonMin, BarraLonMax),
        time=slice(BarraTimeMin, BarraTimeMax)
    )

    # make sure the temperature variable is named the same in each data array
    Tdata = Tdata.rename({Tvariable: "ta"})

    # store that slice away
    TBarraDataStructs[PLevel] = Tdata


# REPEAT THE EXACT SAME THING BUT FOR U-WINDS
# create an empty data structure carrying-bag
UBarraDataStructs = {}
# loop through each pressure level and grab a lat-lon slice over Queensland 
for PLevel in PLevels:
    
    Uvariable = 'ua' + str(PLevel)

    # where the BARRA data live
    UBARRAfolder = f'/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/{Uvariable}/latest/'
    UBARRAfile = Uvariable + f'_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_1hr_{YYYY}{MM}-{YYYY}{MM}.nc'
    UBARRApath = UBARRAfolder + UBARRAfile

    Udata = xr.open_dataset(UBARRApath)

    # take that Radar slice
    Udata = Udata.sel(
        lat=slice(BarraLatMin, BarraLatMax),
        lon=slice(BarraLonMin, BarraLonMax),
        time=slice(BarraTimeMin, BarraTimeMax)
    )

    # make sure the temperature variable is named the same in each data array
    Udata = Udata.rename({Uvariable: "ua"})

    # store that slice away
    UBarraDataStructs[PLevel] = Udata


# REPEAT THE EXACT SAME THING BUT FOR PRESSURE GEOPOTENTIAL HEIGHTS INSTEAD OF TEMPERATURE
# create an empty data structure carrying-bag
VBarraDataStructs = {}
# loop through each pressure level and grab a lat-lon slice over Queensland 
for PLevel in PLevels:
    
    Vvariable = 'va' + str(PLevel)

    # where the BARRA data live
    VBARRAfolder = f'/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/{Vvariable}/latest/'
    VBARRAfile = Vvariable + f'_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_1hr_{YYYY}{MM}-{YYYY}{MM}.nc'
    VBARRApath = VBARRAfolder + VBARRAfile

    Vdata = xr.open_dataset(VBARRApath)

    # take that Radar slice
    Vdata = Vdata.sel(
        lat=slice(BarraLatMin, BarraLatMax),
        lon=slice(BarraLonMin, BarraLonMax),
        time=slice(BarraTimeMin, BarraTimeMax)
    )

    # make sure the temperature variable is named the same in each data array
    Vdata = Vdata.rename({Vvariable: "va"})

    # store that slice away
    VBarraDataStructs[PLevel] = Vdata



# REPEAT THE EXACT SAME THING BUT FOR PRESSURE GEOPOTENTIAL HEIGHTS INSTEAD OF TEMPERATURE
# create an empty data structure carrying-bag
ZBarraDataStructs = {}
# loop through each pressure level and grab a lat-lon slice over Queensland 
for PLevel in PLevels:
    
    Zvariable = 'zg' + str(PLevel)

    # where the BARRA data live
    ZBARRAfolder = f'/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/{Zvariable}/latest/'
    ZBARRAfile = Zvariable + f'_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_1hr_{YYYY}{MM}-{YYYY}{MM}.nc'
    ZBARRApath = ZBARRAfolder + ZBARRAfile

    Zdata = xr.open_dataset(ZBARRApath)

    # take that Radar slice
    Zdata = Zdata.sel(
        lat=slice(BarraLatMin, BarraLatMax),
        lon=slice(BarraLonMin, BarraLonMax),
        time=slice(BarraTimeMin, BarraTimeMax)
    )

    # make sure the temperature variable is named the same in each data array
    Zdata = Zdata.rename({Zvariable: "zg"})

    # store that slice away
    ZBarraDataStructs[PLevel] = Zdata

    
# combine all pressure levels for each variable
Tdatasets = [TBarraDataStructs[PLevel] for PLevel in PLevels]
Udatasets = [UBarraDataStructs[PLevel] for PLevel in PLevels]
Vdatasets = [VBarraDataStructs[PLevel] for PLevel in PLevels]
Zdatasets = [ZBarraDataStructs[PLevel] for PLevel in PLevels]

TGrandData = xr.concat(Tdatasets, dim=xr.DataArray(PLevels, dims="pressure", name="pressure"))
UGrandData = xr.concat(Udatasets, dim=xr.DataArray(PLevels, dims="pressure", name="pressure"))
VGrandData = xr.concat(Vdatasets, dim=xr.DataArray(PLevels, dims="pressure", name="pressure"))
ZGrandData = xr.concat(Zdatasets, dim=xr.DataArray(PLevels, dims="pressure", name="pressure"))

# merge the two into one dataset
GrandData = xr.merge([TGrandData, UGrandData, VGrandData, ZGrandData])

# add attributes to the pressure coordinate
GrandData['pressure'].attrs = {
    'long_name': 'pressure',
    'standard_name': 'pressure',
    'units': 'hPa',
    'axis': 'P'
}


In [ ]:
np.array(xgrid.origin_altitude)

In [ ]:
# ADD ALTITUDE ABOVE SEA LEVEL AS A COORDINATE TO THE RADAR DATA FRAME

# squeeze out the extra dimension from origin_altitude before adding
absolute_altitude = (xgrid['z'] + xgrid['origin_altitude']).squeeze().values

# add the new coordinate to the dataset
xgrid = xgrid.assign_coords(
    altitude=('z', absolute_altitude)
)

# add descriptive attributes to the new coordinate
xgrid['altitude'].attrs = {
    'long_name': 'altitude above sea level',
    'standard_name': 'altitude',
    'units': 'm',
    'description': 'Absolute altitude of each grid level, calculated as grid level height relative to radar origin plus radar origin altitude above sea level'
}


In [ ]:
xgrid

In [ ]:
GrandData

In [ ]:
def interpolate_GrandData_to_radar(xgrid, GrandData, variables=['ta', 'ua', 'va']):
    """
    Interpolates variables from GrandData onto the radar grid (xgrid).

    Parameters
    ----------
    xgrid : xarray.Dataset
        Radar dataset with coordinates: time, lat, lon, altitude (metres above sea level).
        lat and lon are 2D arrays of shape (y, x).
    GrandData : xarray.Dataset
        BARRA dataset with coordinates: time, lat, lon, pressure.
        Must contain a 'zg' variable (geopotential height in metres) and the
        variables listed in the variables argument.
    variables : list of str
        Names of variables in GrandData to interpolate onto xgrid.
        Default is ['ta', 'ua', 'va']. Add more names to this list as needed.

    Returns
    -------
    xgrid : xarray.Dataset
        The radar dataset with the interpolated variables added as new coordinates.
    """

    import numpy as np

    # get the single radar time stamp
    radar_time = xgrid['time'].values[0]

    # find the two nearest time steps in GrandData
    GrandData_times = GrandData['time'].values
    time_diffs = np.abs(GrandData_times - radar_time)
    nearest_time_indices = np.argsort(time_diffs)[:2]
    nearest_times = GrandData_times[nearest_time_indices]

    # compute the time interpolation weight (0 = first time, 1 = second time)
    t0, t1 = nearest_times[0], nearest_times[1]
    if t0 == t1:
        time_weight = 0.0
    else:
        time_weight = (radar_time - t0) / (t1 - t0)
        time_weight = float(time_weight)

    # get the radar grid dimensions
    radar_lats = xgrid['lat'].values    # shape: (y, x)
    radar_lons = xgrid['lon'].values    # shape: (y, x)
    radar_alts = xgrid['altitude'].values  # shape: (41,)

    # get the GrandData spatial grid
    barra_lats = GrandData['lat'].values   # shape: (barra_y,)
    barra_lons = GrandData['lon'].values   # shape: (barra_x,)

    # loop over each variable and interpolate onto the radar grid
    for var in variables:

        print(f'Interpolating {var}...')

        # pull out the two time slices for this variable and for zg
        # shape of each after sel: (pressure, lat, lon)
        var_t0 = GrandData[var].sel(time=t0).values
        var_t1 = GrandData[var].sel(time=t1).values
        zg_t0  = GrandData['zg'].sel(time=t0).values
        zg_t1  = GrandData['zg'].sel(time=t1).values

        # time-interpolate both the variable and zg at each barra lat/lon/pressure point
        var_t = var_t0 + time_weight * (var_t1 - var_t0)  # (pressure, lat, lon)
        zg_t  = zg_t0  + time_weight * (zg_t1  - zg_t0)  # (pressure, lat, lon)

        # prepare an output array to fill: (1, radar_y, radar_x, radar_alt)
        output = np.full(
            (1, radar_lats.shape[0], radar_lats.shape[1], len(radar_alts)),
            np.nan
        )

        # loop over each radar grid point and interpolate in altitude
        for i in range(radar_lats.shape[0]):
            for j in range(radar_lats.shape[1]):

                # get the actual lat/lon value at this grid point
                rlat = radar_lats[i, j]
                rlon = radar_lons[i, j]

                # find the nearest BARRA lat/lon indices
                i_barra = np.argmin(np.abs(barra_lats - rlat))
                j_barra = np.argmin(np.abs(barra_lons - rlon))

                # get the altitude profile and variable profile at this lat/lon
                zg_profile  = zg_t[:, i_barra, j_barra]   # (pressure,)
                var_profile = var_t[:, i_barra, j_barra]   # (pressure,)

                # make sure the profiles are sorted by ascending altitude
                sort_idx    = np.argsort(zg_profile)
                zg_profile  = zg_profile[sort_idx]
                var_profile = var_profile[sort_idx]

                # interpolate the variable onto each radar altitude level
                # radar altitudes below the lowest pressure level get the lowest level value
                output[0, i, j, :] = np.interp(
                    radar_alts,
                    zg_profile,
                    var_profile,
                    left=var_profile[0]   # below lowest level: use lowest level value
                )

        # wrap the output in a DataArray using z as the dimension
        output_da = xr.DataArray(
            output,
            dims=['time', 'y', 'x', 'z'],
            coords={
                'time':     xgrid['time'],
                'altitude': xgrid['altitude']
            }
        )

        # copy over the attributes (long_name, units, etc.) from GrandData
        output_da.attrs = GrandData[var].attrs

        # add the interpolated variable to xgrid with _BARRA suffix
        xgrid[var + '_BARRA'] = output_da

    return xgrid


In [ ]:
interpolate_GrandData_to_radar(xgrid, GrandData)

In [ ]:
xgrid

In [ ]:
print(f'Radar lat shape: {xgrid["lat"].values.shape}')
print(f'Radar lon shape: {xgrid["lon"].values.shape}')



In [ ]:
# Load in a DEM (update path and variable name as needed)
DEMpath = '/home/563/sg3241/QueenslandElevationGEBCO.nc'  # <- your DEM file
DEMdata = xr.open_dataset(DEMpath)
DEMelev = DEMdata['elevation']  # adjust if your var has a different name

In [ ]:
# VERTICAL CROSS SECTION PLOTTING
# (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)

# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# PLEASE MAKE IT SO THE PLOT VARIABLES CAN BE CHOSEN HERE!!!! (ADD STRING EXECUTERS AND SUCH)
# CHOOSE THE VARIABLE TO PLOT
Var  = 'ua_BARRA'

# CHOOSE THE SLICE OF THE CROSS-SECTION
EWsliceKM = 25 # [km]     # number of km north or south of the radar you want to take the east-west slice for x-section
EWsliceNorS = 'North'    # direction ['North' or 'South'] from the radar you want the slice taken

# CHOOSE YOUR ALTITUDE RANGE
MinHeight = 0 # [km] minimum height in plot
MaxHeight = 15 # [km] maximum height in the plot

# THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
# IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES

# Coordinates in km (range from radar)
yvalsKM = np.array(xgrid.x) * 0.001  # x coordinates in km
# farthest south and north y-values
MinSliceKM = int(np.min(yvalsKM))
MaxSliceKM = int(np.max(yvalsKM))

# slice choice follow-on
# THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
# IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES
# THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
# IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES
Slice    = str(EWsliceKM) + 'kmEW'
# add a positive or negative sign to the slice for math
if (EWsliceNorS == 'South'):
    EWsliceKMsign = EWsliceKM * -1
elif (EWsliceNorS == 'North'):
    EWsliceKMsign = EWsliceKM * 1
else:
    print("Please choose EWsliceNorS to be 'North' or 'South'.")

# find the vertical cross section index in the coordinates
EWslicei = np.where(yvalsKM == EWsliceKMsign)[0][0]  # index in the x-coordinates where that north-south km value lives


# radar choice follow-on
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal =  -10 # [dBZ]
    VarMaxVal =   65 # [dBZ]
    VarUnit   = 'dBZ'
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max = 100.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    
elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarUnit   = 'dB'
    VarColourBar = 'RdBu'
    
elif (Var == 'ta_BARRA'):
    VarName     = 'Temperature'
    VarNameLong = 'ta_BARRA'
    VarMinVal =  -40 # [K]
    VarMaxVal =  50 # [K]
    VarUnit   = 'K'
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -40
    VarColourBar_max = 90
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'ua_BARRA'):
    VarName     = 'U-Wind'
    VarNameLong = 'ua_BARRA'
    VarMinVal =  -30 # [m/s]
    VarMaxVal =  35 # [K]
    VarUnit   = 'm/s'
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30
    VarColourBar_max = 35
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'va_BARRA'):
    VarName     = 'V-Wind'
    VarNameLong = 'va_BARRA'
    VarMinVal =  -30 # [m/s]
    VarMaxVal =  35 # [K]
    VarUnit   = 'm/s'
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30
    VarColourBar_max = 35
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
     
elif (Var == 'CC'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarColourBar = 'nipy_spectral'
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = 0  # [deg/ km]
    VarMaxVal = 10 # [deg / km]
    VarUnit   = 'deg / km'
    VarColourBar = 'nipy_spectral'
elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarColourBar = 'nipy_spectral'
elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -30 # [m/s]
    VarMaxVal =  30 # [m/s]
    VarUnit   = 'm/s'
    VarColourBar = 'RdBu_r'
else:
    raise ValueError("Input Variable '" + Var + "' not available\n" + "Please choose from the following list:\n" + \
          "[Z] 'corrected_reflectivity', [CC] 'corrected_cross_correlation_ratio', [ZDR] 'corrected_differential_reflectivity'\n" + \
          "[KDP] 'corrected_specific_differential_phase', [PhiDP] 'corrected_differential_phase'")


# EXPERIMENTAL TOPOGRAPHY SECTION
# EXPERIMENTAL TOPOGRAPHY SECTION
# EXPERIMENTAL TOPOGRAPHY SECTION

# lat/lon fields in xgrid: dimensions (y, x)
# Slice at the chosen north–south index EWslicei along x
lat_slice = xgrid['lat'][EWslicei, :].values  # shape (nx,)
lon_slice = xgrid['lon'][EWslicei, :].values  # shape (nx,)

# Build DataArrays for interpolation
LONdata = xr.DataArray(lon_slice, dims=('x',))
LATdata = xr.DataArray(lat_slice, dims=('x',))

# Interpolate DEM to the cross-section line
dem_slice = DEMelev.interp(lon=LONdata, lat=LATdata)

# Elevation in metres along the line
TerrainSlice = dem_slice.values

# For plotting in km, and do not let negative (ocean) go below 0
TerrainSliceKM = np.maximum(TerrainSlice, 0.0) * 0.001

Xkm = xgrid.x * 0.001  # east–west distance [km]
    

# EXPERIMENTAL TOPOGRAPHY SECTION END
# EXPERIMENTAL TOPOGRAPHY SECTION END
# EXPERIMENTAL TOPOGRAPHY SECTION END
    

# apply the condtional mask to the variable array before plotting
# ValidVariableArray = xgrid[VarNameLong].where(ConditionGrid)
PlottingArray = xgrid[VarNameLong][0,:,EWslicei,:]

# REAL DATA PLOTTING
# REAL DATA PLOTTING
# REAL DATA PLOTTING
fig, ax = plt.subplots(figsize=(8,6))
GridViewer = pcolormeshC(lon_slice, xgrid.z*0.001, PlottingArray.transpose(), ax=ax, cmap=VarColourBar, norm=VarColourBar_norm)
                                         # mutiply by 0.001 to get distances in km

# Plot the line showing ground/topography
ax.plot(lon_slice, TerrainSliceKM, color=[0.3,0.3,0.3], linewidth=1.5, zorder=5 )
ax.fill_between(lon_slice, 0, TerrainSliceKM , color=[0.3,0.3,0.3], zorder=4 ) # Shade everything below the terrain line in black

cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
cbar.ax.set_ylim(VarMinVal, VarMaxVal)          # zoom the displayed range on the colorbar
cbar.set_ticks(np.arange(VarMinVal, VarMaxVal, 10))  # tick every 10 dBZ

# Plot the line showing ground/topography
ax.plot(lon_slice, TerrainSliceKM, color=[0.3,0.3,0.3], linewidth=1.5, zorder=5 )
ax.fill_between(lon_slice, 0, TerrainSliceKM , color=[0.3,0.3,0.3], zorder=4 ) # Shade everything below the terrain line in black


# approximate latitude where the slice is taken (may vary a bit since the radar grid isn't perfect)
ApproxLat = np.round(lat_slice[150], 2)

# change it to positive and write 'north' or 'south'
if (ApproxLat < 0):
    ApproxLat = ApproxLat * -1
    ApproxLatDir = 'S'
else:
    ApproxLatDir = 'N'

ax.set_xlabel('Longitude [Degrees East]')
ax.set_ylabel('Altitude [km]')
plt.title(VarName + ' Cross Section for ' + RadarSiteName + ' Radar\n For Slice Taken ' + \
      str(EWsliceKM) + ' km ' + EWsliceNorS + ' of the Radar (~' + str(ApproxLat) + '° ' + ApproxLatDir + ')\n' + \
      RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
      str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')
plt.grid()

plt.xlim([np.min(lon_slice), np.max(lon_slice)])
plt.ylim([0,MaxHeight+3])

# SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + \
#                                                     RadarFileDate + '/' + VarName + '/RhoHVlim' + str(MinValidRhoHV*100)[0:2] + '/'
#                                                                                                   # RhoHV 0.85 becomes 85 in file name
# SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + VarNameLong + '_' + PlotType + \
# str(EWsliceKM) + EWsliceNorS + '.png'

# SavePath = SaveFolder + SaveFile

# if not Path(SaveFolder).exists():
#     print('Creating Folder: ' + SaveFolder)
#     Path(SaveFolder).mkdir(parents=True, exist_ok=True)

# # plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
# # plt.close()

In [ ]:
print(GrandData['zg'].isnull().any())
print(GrandData['zg'].isnull().sum())

In [ ]:
GrandData['zg'].isnull().sum(dim=['pressure', 'time']).plot()

In [ ]:
GrandData['zg'][:,0,0,0]